Not the most rigorous work and will need an ablation study but one can see there's an incremental improvement on validation loss when using the custom pairwise layer

In [1]:
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchinfo import summary
import torch._dynamo

In [2]:
torch._dynamo.config.cache_size_limit = 16

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device = "cpu"

In [3]:
df = sns.load_dataset('diamonds')
df = df[['carat', 'depth', 'table', 'price', 'x', 'y', 'z', 'cut']]

df['cut'] = (df['cut'] == 'Ideal').astype(int)

X = df.drop('cut', axis=1)
y = df['cut']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long).to(device)

print(X.shape)
print(y.shape)

(53940, 7)
(53940,)


In [4]:
class PairwiseLayer(nn.Module):
    def __init__(self, n: int):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(n, n))
        self.bias   = nn.Parameter(torch.zeros(n, n))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(2).addcmul(x.unsqueeze(1), self.weight).add(self.bias) # (B, n, n)
        
        pos = torch.relu(x)
        neg = torch.relu(-x)

        B = x.size(0)
        return torch.cat([neg.reshape(B, -1), pos.reshape(B, -1)], dim=1) # (B, 2 * n^2)

In [5]:
class DiamondNN(nn.Module):
    def __init__(self, n):
        super().__init__()

        self.n = n
        self.stages = nn.ModuleList()
        self.linears = nn.ModuleList()

        multiplier = 1
        for i in range(2):
            input_dim = n * multiplier
            self.stages.append(PairwiseLayer(input_dim))
            self.linears.append(nn.Linear(2 * input_dim**2, input_dim))

            with torch.no_grad():
                weight = torch.eye(input_dim).repeat_interleave(input_dim, dim=1) / n
                weight = torch.cat([weight, -weight], dim=1)  # shape: (n, 2 * n^2)

                self.linears[i].weight.copy_(weight) # type: ignore
                nn.init.zeros_(self.linears[i].bias) # type: ignore

            multiplier *= 2

        self.output_layer = nn.Linear(n * multiplier, 2)
        nn.init.zeros_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)
        

    def forward(self, x0):
        x = x0
        for stage, linear in zip(self.stages, self.linears):
            x_pre = x
            x = stage(x)
            x = linear(x)
            x = torch.cat((x_pre, x), dim=1)

        return self.output_layer(x)

In [6]:
model = torch.compile(DiamondNN(n=X.shape[1]).to(device))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [7]:
print(summary(model, input_data=X_train_tensor))

torch.set_printoptions(threshold=float('inf'))
for name, param in model.named_parameters():
    print(name)
    print(param)

epochs = 10000
best_test_loss = float('inf')

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = (test_predictions == y_test_tensor).sum().item() / y_test_tensor.size(0)

    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print()
print(f"Lowest Test Loss: {best_test_loss:.4f}")

Layer (type:depth-idx)                   Output Shape              Param #
OptimizedModule                          [43152, 2]                --
├─DiamondNN: 1-1                         [43152, 2]                --
│    └─ModuleList: 2-3                   --                        (recursive)
│    │    └─PairwiseLayer: 3-1           [43152, 98]               98
│    └─ModuleList: 2-4                   --                        (recursive)
│    │    └─Linear: 3-2                  [43152, 7]                693
│    └─ModuleList: 2-3                   --                        (recursive)
│    │    └─PairwiseLayer: 3-3           [43152, 392]              392
│    └─ModuleList: 2-4                   --                        (recursive)
│    │    └─Linear: 3-4                  [43152, 14]               5,502
│    └─Linear: 2-5                       [43152, 2]                58
Total params: 6,743
Trainable params: 6,743
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 290.97
Inpu

In [8]:
class DiamondNN(nn.Module):
    def __init__(self, n):
        super().__init__()

        self.n = n
        self.stages = nn.ModuleList()
        self.linears = nn.ModuleList()

        multiplier = 1
        for _ in range(3):
            input_dim = n * multiplier
            self.stages.append(nn.Linear(input_dim, input_dim * 3))
            self.linears.append(nn.Linear(3 * input_dim, input_dim))
            multiplier *= 2

        self.output_layer = nn.Linear(n * multiplier, 2)

    def forward(self, x):
        for stage, linear in zip(self.stages, self.linears):
            x_pre = x

            x = torch.relu(stage(x))
            x = torch.relu(linear(x))
            
            x = torch.cat((x_pre, x), dim=1)

        return self.output_layer(x)

In [9]:
model = torch.compile(DiamondNN(n=X.shape[1]).to(device))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [10]:
print(summary(model, input_data=X_train_tensor[:1]))

torch.set_printoptions(threshold=float('inf'))
for name, param in model.named_parameters():
    print(name)
    print(param)

epochs = 10000
best_test_loss = float('inf')

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = (test_predictions == y_test_tensor).sum().item() / y_test_tensor.size(0)

    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print()
print(f"Lowest Test Loss: {best_test_loss:.4f}")

Layer (type:depth-idx)                   Output Shape              Param #
OptimizedModule                          [1, 2]                    --
├─DiamondNN: 1-1                         [1, 2]                    --
│    └─ModuleList: 2-5                   --                        (recursive)
│    │    └─Linear: 3-1                  [1, 21]                   168
│    └─ModuleList: 2-6                   --                        (recursive)
│    │    └─Linear: 3-2                  [1, 7]                    154
│    └─ModuleList: 2-5                   --                        (recursive)
│    │    └─Linear: 3-3                  [1, 42]                   630
│    └─ModuleList: 2-6                   --                        (recursive)
│    │    └─Linear: 3-4                  [1, 14]                   602
│    └─ModuleList: 2-5                   --                        (recursive)
│    │    └─Linear: 3-5                  [1, 84]                   2,436
│    └─ModuleList: 2-6           